# Data Preparation for Clustering and PCA

**Dataset:** CIC-IDS2017 (cleaned)  
**Output:** `cluster_sample_unlabeled.csv`, `cluster_sample_labels.csv`

Clustering and PCA operate on unlabeled, numeric features. This notebook derives that dataset from the cleaned CIC-IDS2017 flow records.

In [10]:
import numpy as np
import pandas as pd

## 1. Load Data

In [11]:
df = pd.read_csv("cicids2017_cleaned.csv", low_memory=False)
print(f"Rows: {df.shape[0]:,}  Columns: {df.shape[1]}")

Rows: 2,520,798  Columns: 79


## 2. Separate Labels and Select Numeric Features

- `Label` is held out, since clustering and PCA are unsupervised.
- `Destination Port` is excluded. It is a categorical identifier stored as an integer, so distance calculations on it are not meaningful.
- Infinite values are converted to missing values.

In [12]:
labels = df["Label"]
X = df.drop(columns=["Label", "Destination Port"])
X = X.select_dtypes(include="number")
X = X.replace([np.inf, -np.inf], np.nan)

print(f"Numeric features: {X.shape[1]}")
print(f"Missing values:   {int(X.isna().sum().sum())}")

Numeric features: 77
Missing values:   0


## 3. Remove Constant Features

Features with a single unique value have zero variance and cannot be standardized.

In [13]:
const_cols = list(X.columns[X.nunique() <= 1])
print(f"Constant features removed ({len(const_cols)}): {const_cols}")

X = X.drop(columns=const_cols)
print(f"Features remaining: {X.shape[1]}")

Constant features removed (8): ['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']
Features remaining: 69


## 4. Remove All-Zero Rows

Cosine distance is undefined for zero vectors.

In [14]:
nonzero = (X != 0).any(axis=1)
print(f"All-zero rows removed: {(~nonzero).sum():,}")

X = X[nonzero]
labels = labels[nonzero]
print(f"Rows remaining: {len(X):,}")

All-zero rows removed: 0
Rows remaining: 2,520,798


## 5. Stratified Sample

Hierarchical clustering does not scale to millions of rows. A random sample would be roughly 83% benign traffic and would contain few or no rare attack types. Each class is therefore capped at 1,000 rows; classes with fewer rows are retained in full.

The resulting sample is more balanced than the source data.

In [15]:
CAP = 1000
SEED = 42

parts = [
    labels[labels == lab].sample(min((labels == lab).sum(), CAP), random_state=SEED)
    for lab in labels.unique()
]

idx = pd.concat(parts).index
sample_X = X.loc[idx].sample(frac=1, random_state=SEED)
sample_y = labels.loc[sample_X.index]

print(f"Sample shape: {sample_X.shape}")
sample_y.value_counts()

Sample shape: (11720, 69)


Label
BENIGN                        1000
SSH-Patator                   1000
DoS GoldenEye                 1000
Bot                           1000
DDoS                          1000
Web Attack - Brute Force      1000
PortScan                      1000
DoS Hulk                      1000
FTP-Patator                   1000
DoS slowloris                 1000
DoS Slowhttptest              1000
Web Attack - XSS               652
Infiltration                    36
Web Attack - Sql Injection      21
Heartbleed                      11
Name: count, dtype: int64

## 6. Remove Features Constant Within the Sample

Some features vary in the full dataset but are constant in the sample. They carry no information here and are removed.

In [16]:
sample_const = list(sample_X.columns[sample_X.nunique() <= 1])
print(f"Sample-constant features removed ({len(sample_const)}): {sample_const}")

sample_X = sample_X.drop(columns=sample_const)
print(f"Final shape: {sample_X.shape}")

Sample-constant features removed (4): ['Fwd URG Flags', 'RST Flag Count', 'CWE Flag Count', 'ECE Flag Count']
Final shape: (11720, 65)


## 7. Save Outputs

The two files share the same row order. The unlabeled file is the input to clustering and PCA; the label file is used only to evaluate results.

In [17]:
sample_X.to_csv("cluster_sample_unlabeled.csv", index=False)
sample_y.to_csv("cluster_sample_labels.csv", index=False)
print("Saved: cluster_sample_unlabeled.csv, cluster_sample_labels.csv")

Saved: cluster_sample_unlabeled.csv, cluster_sample_labels.csv


## 8. Sample Preview

In [18]:
preview_cols = [
    "Flow Duration", "Total Fwd Packets", "Total Backward Packets",
    "Fwd Packet Length Mean", "Flow Bytes/s", "Flow Packets/s",
]
sample_X[[c for c in preview_cols if c in sample_X.columns]].head()

,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packet Length Mean,Flow Bytes/s,Flow Packets/s
1136918,31420,2,2,38.000000,8147.676639,127.307447
1549100,12053958,21,32,95.619048,394.310317,4.396896
1645578,14456137,22,33,90.545455,327.680901,3.804613
2333520,11875861,7,5,41.714286,1004.053517,1.010453
838862,54,1,1,6.000000,222222.222200,37037.037040
